# 3.3 Multi-class Classification

使用 Wine Dataset 建立表格資料多類別分類 DNN。這份 notebook 示範資料切分、標準化、softmax 輸出層、`sparse_categorical_crossentropy`、confusion matrix、classification report 與新資料預測。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

## 1. 載入 Wine Dataset

Wine Dataset 是 scikit-learn 內建的小型表格資料集。每筆資料是一款葡萄酒的化學分析結果，目標欄位 `target` 代表葡萄酒類別。這份資料有 13 個數值特徵與 3 個互斥類別，很適合用來示範多類別分類。

In [ ]:
data = load_wine(as_frame=True)
df = data.frame
print(df.shape)
df.head()

## 2. 切分資料與標準化

多類別分類仍要先固定 train/test split。這裡使用 `stratify=y` 讓訓練集與測試集保留相近的類別比例。

表格數值特徵尺度可能差很多，因此使用 `StandardScaler`。注意 `fit_transform()` 只用在訓練集，測試集只能使用同一個 scaler 做 `transform()`。

In [ ]:
X = df.drop(columns=['target']).values
y = df['target'].values
num_classes = len(np.unique(y))

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(num_classes, x_train.shape, x_test.shape)

## 3. 建立 DNN 多類別分類模型

多類別分類的最後一層使用 `Dense(num_classes, activation='softmax')`。Wine Dataset 的標籤是整數類別，因此 loss 使用 `sparse_categorical_crossentropy`。如果標籤已經是 one-hot，才改用 `categorical_crossentropy`。

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. 訓練模型

這裡使用 `EarlyStopping` 監控 validation loss。當 validation loss 一段時間沒有改善時停止訓練，並透過 `restore_best_weights=True` 回復到 validation loss 最好的模型權重。

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0
)

pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Accuracy Curve')
plt.show()

## 5. 評估模型

先比較 train/test loss 與 accuracy，確認模型是否只記住訓練資料。接著使用 confusion matrix 與 classification report 觀察每個類別的 precision、recall 與 F1-score。

In [ ]:
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

pd.DataFrame({
    'dataset': ['train', 'test'],
    'loss': [train_loss, test_loss],
    'accuracy': [train_acc, test_acc]
})

In [ ]:
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 6. 預測新資料

`model.predict()` 回傳每個類別的機率。多類別分類通常用 `argmax` 取得機率最高的類別，並可同時保存最高機率作為 confidence。

In [ ]:
sample_prob = model.predict(x_test[:5], verbose=0)
pd.DataFrame({
    'predicted_class': np.argmax(sample_prob, axis=1),
    'confidence': np.max(sample_prob, axis=1),
    'true_label': y_test[:5]
})